In [6]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.extract import load_config, extract_csv

In [7]:
config = load_config()

df = extract_csv(config["data"]["raw_path"])

print(df.shape)

df.head()

(496615, 17)


,Unnamed: 0,Organization Full Name,Organization Class,Responsible Party,Brief Title,Full Title,Overall Status,Start Date,Standard Age,Conditions,Primary Purpose,Interventions,Intervention Description,Study Type,Phases,Outcome Measure,Medical Subject Headings
0,0,Montefiore Medical Center,OTHER,SPONSOR,Kinesiotape for Edema After Bilateral Total Kn...,"Effect of Kinesiotaping on Edema Management, P...",COMPLETED,2021-10-18,ADULT OLDER_ADULT,"Arthroplasty Complications, Arthroplasty, Repl...",TREATMENT,Kinesio(R)Tape for edema control,"Kinesio(R)Tape is an elastic, cotton tape with...",INTERVENTIONAL,NaN,Change from baseline and during 1-2-day time i...,Edema
1,1,Seoul National University Hospital,OTHER,Unknown,An Open-labeled Trial of Ramipril in Patients ...,An Open-labeled Trial of Ramipril in Patients ...,COMPLETED,2004-10,ADULT OLDER_ADULT,Migraine With Hypertension,TREATMENT,Ramipril,ramipril 2.5mg twice a day,INTERVENTIONAL,PHASE2,headache frequency,Migraine Disorders Hypertension
2,2,Federico II University,OTHER,PRINCIPAL_INVESTIGATOR,OCTA in Epivascular Glia After Dex Implant,Evaluation of Changes in Epivascular Glia Befo...,COMPLETED,2021-01-01,ADULT OLDER_ADULT,Diabetic Retinopathy,Unknown,Dexamethasone intravitreal implant,Dexamethasone intravitreal implant,OBSERVATIONAL,Unknown,Changes in epivascular glia after Dexamethason...,Diabetic Retinopathy
3,3,Sidney Kimmel Comprehensive Cancer Center at J...,OTHER,SPONSOR,Preoperative Immune Checkpoint Inhibitor for P...,Preoperative Immune Checkpoint Inhibitor Thera...,COMPLETED,2019-07-08,ADULT OLDER_ADULT,"Head and Neck Squamous Cell Carcinoma, Head an...",TREATMENT,Nivolumab 480mg and surgical resection,One dose of Nivolumab 480mg given four weeks p...,INTERVENTIONAL,PHASE2,Safety as measured by number of participants w...,"Carcinoma Carcinoma, Squamous Cell Head and Ne..."
4,4,Northwestern University,OTHER,SPONSOR,Genistein in Treating Patients With Prostate C...,Phase 2 Trial of Genistein in Men With Circula...,TERMINATED,2011-02-03,ADULT OLDER_ADULT,"Adenocarcinoma of the Prostate, Recurrent Pros...",TREATMENT,"genistein, placebo, therapeutic conventional s...","Given orally, Given orally, Radical prostatect...",INTERVENTIONAL,PHASE2,Number of Circulating Prostate Cells (CPCs) in...,Prostatic Neoplasms


In [8]:
import pandas as pd


def drop_index_column(df):
    df = df.copy()

    unnamed_cols = [col for col in df.columns if col.startswith("Unnamed")]

    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    return df


def cast_column_types(df):
    df = df.copy()

    df["Start Date"] = pd.to_datetime(
        df["Start Date"],
        format="%Y-%m",
        errors="coerce"
    )

    string_columns = [
        "Organization Full Name",
        "Organization Class",
        "Responsible Party",
        "Brief Title",
        "Full Title",
        "Overall Status",
        "Standard Age",
        "Conditions",
        "Primary Purpose",
        "Interventions",
        "Intervention Description",
        "Study Type",
        "Phases",
        "Outcome Measure",
        "Medical Subject Headings",
    ]

    for column in string_columns:
        df[column] = df[column].astype("string")

    return df


def separate_withheld_rows(df):
    df = df.copy()

    withheld_mask = (
        df["Organization Full Name"]
        .str.strip()
        .eq("[Redacted]")
    )

    withheld_df = df.loc[withheld_mask].copy()
    clean_df = df.loc[~withheld_mask].copy()

    return clean_df, withheld_df


def remove_duplicate_studies(df): 
    return df.drop_duplicates(ignore_index=True)
   
    ## business duplicate definition needs to be understood/ defined from domain info (REVIEW!)
    # df = df.copy()

    # duplicate_columns = [
    #     "Brief Title",
    #     "Organization Full Name",
    #     "Full Title",
    #     "Start Date",
    #     "Interventions",
    #     "Conditions",
    #     "Phases",
    #     "Outcome Measure",
    # ]

    # df = df.drop_duplicates(
    #     subset=duplicate_columns,
    #     keep="first"
    # )

    return df


def add_withheld_flag(clean_df, withheld_df):
    clean_df = clean_df.copy()
    withheld_df = withheld_df.copy()

    clean_df["is_withheld"] = False
    withheld_df["is_withheld"] = True

    return pd.concat(
        [clean_df, withheld_df],
        ignore_index=True
    )


def add_study_id(df):
    df = df.copy()

    df.insert(
        0,
        "study_id",
        range(1, len(df) + 1)
    )

    return df


def transform(df):

    df = drop_index_column(df)

    df = cast_column_types(df)

    clean_df, withheld_df = separate_withheld_rows(df)

    clean_df = remove_duplicate_studies(clean_df)

    df = add_withheld_flag(
        clean_df,
        withheld_df
    )

    df = add_study_id(df)

    return df

In [9]:
df1 = drop_index_column(df)

print(df1.shape)
df1.head()

(496615, 16)


,Organization Full Name,Organization Class,Responsible Party,Brief Title,Full Title,Overall Status,Start Date,Standard Age,Conditions,Primary Purpose,Interventions,Intervention Description,Study Type,Phases,Outcome Measure,Medical Subject Headings
0,Montefiore Medical Center,OTHER,SPONSOR,Kinesiotape for Edema After Bilateral Total Kn...,"Effect of Kinesiotaping on Edema Management, P...",COMPLETED,2021-10-18,ADULT OLDER_ADULT,"Arthroplasty Complications, Arthroplasty, Repl...",TREATMENT,Kinesio(R)Tape for edema control,"Kinesio(R)Tape is an elastic, cotton tape with...",INTERVENTIONAL,NaN,Change from baseline and during 1-2-day time i...,Edema
1,Seoul National University Hospital,OTHER,Unknown,An Open-labeled Trial of Ramipril in Patients ...,An Open-labeled Trial of Ramipril in Patients ...,COMPLETED,2004-10,ADULT OLDER_ADULT,Migraine With Hypertension,TREATMENT,Ramipril,ramipril 2.5mg twice a day,INTERVENTIONAL,PHASE2,headache frequency,Migraine Disorders Hypertension
2,Federico II University,OTHER,PRINCIPAL_INVESTIGATOR,OCTA in Epivascular Glia After Dex Implant,Evaluation of Changes in Epivascular Glia Befo...,COMPLETED,2021-01-01,ADULT OLDER_ADULT,Diabetic Retinopathy,Unknown,Dexamethasone intravitreal implant,Dexamethasone intravitreal implant,OBSERVATIONAL,Unknown,Changes in epivascular glia after Dexamethason...,Diabetic Retinopathy
3,Sidney Kimmel Comprehensive Cancer Center at J...,OTHER,SPONSOR,Preoperative Immune Checkpoint Inhibitor for P...,Preoperative Immune Checkpoint Inhibitor Thera...,COMPLETED,2019-07-08,ADULT OLDER_ADULT,"Head and Neck Squamous Cell Carcinoma, Head an...",TREATMENT,Nivolumab 480mg and surgical resection,One dose of Nivolumab 480mg given four weeks p...,INTERVENTIONAL,PHASE2,Safety as measured by number of participants w...,"Carcinoma Carcinoma, Squamous Cell Head and Ne..."
4,Northwestern University,OTHER,SPONSOR,Genistein in Treating Patients With Prostate C...,Phase 2 Trial of Genistein in Men With Circula...,TERMINATED,2011-02-03,ADULT OLDER_ADULT,"Adenocarcinoma of the Prostate, Recurrent Pros...",TREATMENT,"genistein, placebo, therapeutic conventional s...","Given orally, Given orally, Radical prostatect...",INTERVENTIONAL,PHASE2,Number of Circulating Prostate Cells (CPCs) in...,Prostatic Neoplasms


In [10]:
df2 = cast_column_types(df1)

df2.dtypes

Organization Full Name              string
Organization Class                  string
Responsible Party                   string
Brief Title                         string
Full Title                          string
Overall Status                      string
Start Date                  datetime64[us]
Standard Age                        string
Conditions                          string
Primary Purpose                     string
Interventions                       string
Intervention Description            string
Study Type                          string
Phases                              string
Outcome Measure                     string
Medical Subject Headings            string
dtype: object

In [11]:
clean_df, withheld_df = separate_withheld_rows(df2)

print(clean_df.shape)
print(withheld_df.shape)

(495730, 16)
(885, 16)


In [12]:
clean_df = remove_duplicate_studies(clean_df)

print(clean_df.shape)

(495694, 16)


In [13]:
combined_df = add_withheld_flag(
    clean_df,
    withheld_df
)

combined_df["is_withheld"].value_counts()

is_withheld
False    495694
True        885
Name: count, dtype: int64

In [14]:
combined_df = add_study_id(combined_df)

combined_df.head()

,study_id,Organization Full Name,Organization Class,Responsible Party,Brief Title,Full Title,Overall Status,Start Date,Standard Age,Conditions,Primary Purpose,Interventions,Intervention Description,Study Type,Phases,Outcome Measure,Medical Subject Headings,is_withheld
0,1,Montefiore Medical Center,OTHER,SPONSOR,Kinesiotape for Edema After Bilateral Total Kn...,"Effect of Kinesiotaping on Edema Management, P...",COMPLETED,NaT,ADULT OLDER_ADULT,"Arthroplasty Complications, Arthroplasty, Repl...",TREATMENT,Kinesio(R)Tape for edema control,"Kinesio(R)Tape is an elastic, cotton tape with...",INTERVENTIONAL,<NA>,Change from baseline and during 1-2-day time i...,Edema,False
1,2,Seoul National University Hospital,OTHER,Unknown,An Open-labeled Trial of Ramipril in Patients ...,An Open-labeled Trial of Ramipril in Patients ...,COMPLETED,2004-10-01,ADULT OLDER_ADULT,Migraine With Hypertension,TREATMENT,Ramipril,ramipril 2.5mg twice a day,INTERVENTIONAL,PHASE2,headache frequency,Migraine Disorders Hypertension,False
2,3,Federico II University,OTHER,PRINCIPAL_INVESTIGATOR,OCTA in Epivascular Glia After Dex Implant,Evaluation of Changes in Epivascular Glia Befo...,COMPLETED,NaT,ADULT OLDER_ADULT,Diabetic Retinopathy,Unknown,Dexamethasone intravitreal implant,Dexamethasone intravitreal implant,OBSERVATIONAL,Unknown,Changes in epivascular glia after Dexamethason...,Diabetic Retinopathy,False
3,4,Sidney Kimmel Comprehensive Cancer Center at J...,OTHER,SPONSOR,Preoperative Immune Checkpoint Inhibitor for P...,Preoperative Immune Checkpoint Inhibitor Thera...,COMPLETED,NaT,ADULT OLDER_ADULT,"Head and Neck Squamous Cell Carcinoma, Head an...",TREATMENT,Nivolumab 480mg and surgical resection,One dose of Nivolumab 480mg given four weeks p...,INTERVENTIONAL,PHASE2,Safety as measured by number of participants w...,"Carcinoma Carcinoma, Squamous Cell Head and Ne...",False
4,5,Northwestern University,OTHER,SPONSOR,Genistein in Treating Patients With Prostate C...,Phase 2 Trial of Genistein in Men With Circula...,TERMINATED,NaT,ADULT OLDER_ADULT,"Adenocarcinoma of the Prostate, Recurrent Pros...",TREATMENT,"genistein, placebo, therapeutic conventional s...","Given orally, Given orally, Radical prostatect...",INTERVENTIONAL,PHASE2,Number of Circulating Prostate Cells (CPCs) in...,Prostatic Neoplasms,False


In [15]:
final_df = transform(df)

print(final_df.shape)

final_df.head()

(496579, 18)


,study_id,Organization Full Name,Organization Class,Responsible Party,Brief Title,Full Title,Overall Status,Start Date,Standard Age,Conditions,Primary Purpose,Interventions,Intervention Description,Study Type,Phases,Outcome Measure,Medical Subject Headings,is_withheld
0,1,Montefiore Medical Center,OTHER,SPONSOR,Kinesiotape for Edema After Bilateral Total Kn...,"Effect of Kinesiotaping on Edema Management, P...",COMPLETED,NaT,ADULT OLDER_ADULT,"Arthroplasty Complications, Arthroplasty, Repl...",TREATMENT,Kinesio(R)Tape for edema control,"Kinesio(R)Tape is an elastic, cotton tape with...",INTERVENTIONAL,<NA>,Change from baseline and during 1-2-day time i...,Edema,False
1,2,Seoul National University Hospital,OTHER,Unknown,An Open-labeled Trial of Ramipril in Patients ...,An Open-labeled Trial of Ramipril in Patients ...,COMPLETED,2004-10-01,ADULT OLDER_ADULT,Migraine With Hypertension,TREATMENT,Ramipril,ramipril 2.5mg twice a day,INTERVENTIONAL,PHASE2,headache frequency,Migraine Disorders Hypertension,False
2,3,Federico II University,OTHER,PRINCIPAL_INVESTIGATOR,OCTA in Epivascular Glia After Dex Implant,Evaluation of Changes in Epivascular Glia Befo...,COMPLETED,NaT,ADULT OLDER_ADULT,Diabetic Retinopathy,Unknown,Dexamethasone intravitreal implant,Dexamethasone intravitreal implant,OBSERVATIONAL,Unknown,Changes in epivascular glia after Dexamethason...,Diabetic Retinopathy,False
3,4,Sidney Kimmel Comprehensive Cancer Center at J...,OTHER,SPONSOR,Preoperative Immune Checkpoint Inhibitor for P...,Preoperative Immune Checkpoint Inhibitor Thera...,COMPLETED,NaT,ADULT OLDER_ADULT,"Head and Neck Squamous Cell Carcinoma, Head an...",TREATMENT,Nivolumab 480mg and surgical resection,One dose of Nivolumab 480mg given four weeks p...,INTERVENTIONAL,PHASE2,Safety as measured by number of participants w...,"Carcinoma Carcinoma, Squamous Cell Head and Ne...",False
4,5,Northwestern University,OTHER,SPONSOR,Genistein in Treating Patients With Prostate C...,Phase 2 Trial of Genistein in Men With Circula...,TERMINATED,NaT,ADULT OLDER_ADULT,"Adenocarcinoma of the Prostate, Recurrent Pros...",TREATMENT,"genistein, placebo, therapeutic conventional s...","Given orally, Given orally, Radical prostatect...",INTERVENTIONAL,PHASE2,Number of Circulating Prostate Cells (CPCs) in...,Prostatic Neoplasms,False


## testing

In [16]:
# %%
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from src.extract import load_config, extract_csv
from src.transform import transform

In [17]:
# %%
config = load_config()

df = extract_csv(
    config["data"]["raw_path"]
)

print(df.shape)

df.head()

(496615, 17)


,Unnamed: 0,Organization Full Name,Organization Class,Responsible Party,Brief Title,Full Title,Overall Status,Start Date,Standard Age,Conditions,Primary Purpose,Interventions,Intervention Description,Study Type,Phases,Outcome Measure,Medical Subject Headings
0,0,Montefiore Medical Center,OTHER,SPONSOR,Kinesiotape for Edema After Bilateral Total Kn...,"Effect of Kinesiotaping on Edema Management, P...",COMPLETED,2021-10-18,ADULT OLDER_ADULT,"Arthroplasty Complications, Arthroplasty, Repl...",TREATMENT,Kinesio(R)Tape for edema control,"Kinesio(R)Tape is an elastic, cotton tape with...",INTERVENTIONAL,NaN,Change from baseline and during 1-2-day time i...,Edema
1,1,Seoul National University Hospital,OTHER,Unknown,An Open-labeled Trial of Ramipril in Patients ...,An Open-labeled Trial of Ramipril in Patients ...,COMPLETED,2004-10,ADULT OLDER_ADULT,Migraine With Hypertension,TREATMENT,Ramipril,ramipril 2.5mg twice a day,INTERVENTIONAL,PHASE2,headache frequency,Migraine Disorders Hypertension
2,2,Federico II University,OTHER,PRINCIPAL_INVESTIGATOR,OCTA in Epivascular Glia After Dex Implant,Evaluation of Changes in Epivascular Glia Befo...,COMPLETED,2021-01-01,ADULT OLDER_ADULT,Diabetic Retinopathy,Unknown,Dexamethasone intravitreal implant,Dexamethasone intravitreal implant,OBSERVATIONAL,Unknown,Changes in epivascular glia after Dexamethason...,Diabetic Retinopathy
3,3,Sidney Kimmel Comprehensive Cancer Center at J...,OTHER,SPONSOR,Preoperative Immune Checkpoint Inhibitor for P...,Preoperative Immune Checkpoint Inhibitor Thera...,COMPLETED,2019-07-08,ADULT OLDER_ADULT,"Head and Neck Squamous Cell Carcinoma, Head an...",TREATMENT,Nivolumab 480mg and surgical resection,One dose of Nivolumab 480mg given four weeks p...,INTERVENTIONAL,PHASE2,Safety as measured by number of participants w...,"Carcinoma Carcinoma, Squamous Cell Head and Ne..."
4,4,Northwestern University,OTHER,SPONSOR,Genistein in Treating Patients With Prostate C...,Phase 2 Trial of Genistein in Men With Circula...,TERMINATED,2011-02-03,ADULT OLDER_ADULT,"Adenocarcinoma of the Prostate, Recurrent Pros...",TREATMENT,"genistein, placebo, therapeutic conventional s...","Given orally, Given orally, Radical prostatect...",INTERVENTIONAL,PHASE2,Number of Circulating Prostate Cells (CPCs) in...,Prostatic Neoplasms


In [18]:
# %%
final_df = transform(df)

print(final_df.shape)

final_df.head()

(495695, 17)


,Organization Full Name,Organization Class,Responsible Party,Brief Title,Full Title,Overall Status,Start Date,Standard Age,Conditions,Primary Purpose,Interventions,Intervention Description,Study Type,Phases,Outcome Measure,Medical Subject Headings,is_withheld
0,Montefiore Medical Center,OTHER,SPONSOR,Kinesiotape for Edema After Bilateral Total Kn...,"Effect of Kinesiotaping on Edema Management, P...",COMPLETED,NaT,ADULT OLDER_ADULT,"Arthroplasty Complications, Arthroplasty, Repl...",TREATMENT,Kinesio(R)Tape for edema control,"Kinesio(R)Tape is an elastic, cotton tape with...",INTERVENTIONAL,<NA>,Change from baseline and during 1-2-day time i...,Edema,False
1,Seoul National University Hospital,OTHER,Unknown,An Open-labeled Trial of Ramipril in Patients ...,An Open-labeled Trial of Ramipril in Patients ...,COMPLETED,2004-10-01,ADULT OLDER_ADULT,Migraine With Hypertension,TREATMENT,Ramipril,ramipril 2.5mg twice a day,INTERVENTIONAL,PHASE2,headache frequency,Migraine Disorders Hypertension,False
2,Federico II University,OTHER,PRINCIPAL_INVESTIGATOR,OCTA in Epivascular Glia After Dex Implant,Evaluation of Changes in Epivascular Glia Befo...,COMPLETED,NaT,ADULT OLDER_ADULT,Diabetic Retinopathy,Unknown,Dexamethasone intravitreal implant,Dexamethasone intravitreal implant,OBSERVATIONAL,Unknown,Changes in epivascular glia after Dexamethason...,Diabetic Retinopathy,False
3,Sidney Kimmel Comprehensive Cancer Center at J...,OTHER,SPONSOR,Preoperative Immune Checkpoint Inhibitor for P...,Preoperative Immune Checkpoint Inhibitor Thera...,COMPLETED,NaT,ADULT OLDER_ADULT,"Head and Neck Squamous Cell Carcinoma, Head an...",TREATMENT,Nivolumab 480mg and surgical resection,One dose of Nivolumab 480mg given four weeks p...,INTERVENTIONAL,PHASE2,Safety as measured by number of participants w...,"Carcinoma Carcinoma, Squamous Cell Head and Ne...",False
4,Northwestern University,OTHER,SPONSOR,Genistein in Treating Patients With Prostate C...,Phase 2 Trial of Genistein in Men With Circula...,TERMINATED,NaT,ADULT OLDER_ADULT,"Adenocarcinoma of the Prostate, Recurrent Pros...",TREATMENT,"genistein, placebo, therapeutic conventional s...","Given orally, Given orally, Radical prostatect...",INTERVENTIONAL,PHASE2,Number of Circulating Prostate Cells (CPCs) in...,Prostatic Neoplasms,False


In [19]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 495695 entries, 0 to 495694
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Organization Full Name    495695 non-null  string        
 1   Organization Class        495695 non-null  string        
 2   Responsible Party         495695 non-null  string        
 3   Brief Title               495695 non-null  string        
 4   Full Title                495695 non-null  string        
 5   Overall Status            495695 non-null  string        
 6   Start Date                214964 non-null  datetime64[us]
 7   Standard Age              495694 non-null  string        
 8   Conditions                495695 non-null  string        
 9   Primary Purpose           495695 non-null  string        
 10  Interventions             495695 non-null  string        
 11  Intervention Description  495695 non-null  string        
 12  Study Type   

testing

In [20]:
one_row = final_df.head(1)

one_row

,Organization Full Name,Organization Class,Responsible Party,Brief Title,Full Title,Overall Status,Start Date,Standard Age,Conditions,Primary Purpose,Interventions,Intervention Description,Study Type,Phases,Outcome Measure,Medical Subject Headings,is_withheld
0,Montefiore Medical Center,OTHER,SPONSOR,Kinesiotape for Edema After Bilateral Total Kn...,"Effect of Kinesiotaping on Edema Management, P...",COMPLETED,NaT,ADULT OLDER_ADULT,"Arthroplasty Complications, Arthroplasty, Repl...",TREATMENT,Kinesio(R)Tape for edema control,"Kinesio(R)Tape is an elastic, cotton tape with...",INTERVENTIONAL,<NA>,Change from baseline and during 1-2-day time i...,Edema,False


In [21]:
from src.load import load_studies

NameError: name 'df' is not defined